# Neural Network V9

In V4 the best image network had two hidden layers of 1024 neurons, but it reached 100% training accuracy and only 39% validation accuracy, so it was overfitting a lot. For this experiment I tested dropout and weight decay on that network to see if regularization helps. I only did the image dataset, because I already tuned these for sentiment in V3.

First, I imported the libraries and functions I'll need.

In [1]:
from pathlib import Path
from dataclasses import replace
import sys
import time

import numpy as np
import pandas as pd
import torch
from torch import nn

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.data_loader import load_task, split_train_val, standardize
from src.neural_net import ModelConfig, evaluate, parameter_count, predict_probabilities, train_model

device = torch.device('cpu')
EPOCHS = 20
SEED = 46  # the seed V4 used

X, y, _, _ = load_task('image', data_dir=ROOT / 'data')
X_train, X_val, y_train, y_val = split_train_val(X, y, val_fraction=0.2, seed=42)
X_train, X_val = standardize(X_train, X_val)
print(X_train.shape, X_val.shape)

(16000, 500) (4000, 500)


These are the settings V4 selected for image. Every run below changes only dropout or only weight decay.

In [2]:
v4_config = ModelConfig(
    hidden_sizes=(1024, 1024),
    optimizer='adam',
    learning_rate=0.001,
    batch_size=256,
    dropout=0.0,
    weight_decay=0.001,
)

runs = {}


def run(config):
    key = (config.dropout, config.weight_decay)
    if key in runs:
        return runs[key]
    started = time.perf_counter()
    result = train_model(
        X_train, y_train, 50, config, EPOCHS, seed=SEED, device=device,
        validation_features=X_val, validation_labels=y_val,
    )
    training_time = time.perf_counter() - started
    _, training_accuracy = evaluate(
        result.model, X_train, y_train, config.batch_size, device, nn.CrossEntropyLoss()
    )
    predictions = predict_probabilities(result.model, X_val, config.batch_size, device).argmax(axis=1)
    runs[key] = {
        'result': result,
        'predictions': predictions,
        'validation_accuracy': float(np.mean(predictions == y_val)),
        'training_accuracy': float(training_accuracy),
        'training_time': training_time,
    }
    return runs[key]


baseline = run(v4_config)
print(f"V4 settings again: validation accuracy {baseline['validation_accuracy']:.5f}")

V4 settings again: validation accuracy 0.38875


I got the same accuracy as V4, so the comparison starts from the same model. Next I tried dropout with the weight decay kept at 0.001.

In [3]:
rows = []


def add_row(factor, value, config):
    out = run(config)
    rows.append({
        'version': 9,
        'task': 'image',
        'factor': factor,
        'value': value,
        'dropout': config.dropout,
        'weight_decay': config.weight_decay,
        'epochs': EPOCHS,
        'best_epoch': out['result'].best_epoch,
        'config': config.description(),
        'parameters': parameter_count(out['result'].model),
        'training_samples': len(X_train),
        'validation_samples': len(X_val),
        'training_time_seconds': out['training_time'],
        'best_val_accuracy': out['validation_accuracy'],
        'last_epoch_val_accuracy': float(out['result'].history[-1]['val_accuracy']),
        'final_train_accuracy': out['training_accuracy'],
        'train_val_gap': out['training_accuracy'] - out['validation_accuracy'],
    })


columns = ['factor', 'value', 'best_epoch', 'final_train_accuracy', 'best_val_accuracy',
           'last_epoch_val_accuracy', 'training_time_seconds']

for value in (0.0, 0.2, 0.5):
    add_row('dropout', value, replace(v4_config, dropout=value))
print(pd.DataFrame(rows)[columns].to_string(index=False))

 factor  value  best_epoch  final_train_accuracy  best_val_accuracy  last_epoch_val_accuracy  training_time_seconds
dropout    0.0          10              1.000000            0.38875                  0.36875              22.540220
dropout    0.2          17              0.999062            0.37525                  0.36925              26.053072
dropout    0.5          20              0.996500            0.38225                  0.38225              29.742439


Then I tried different weight decay values with no dropout.

In [4]:
for value in (0.0, 0.0001, 0.001, 0.01):
    add_row('weight_decay', value, replace(v4_config, weight_decay=value))
print(pd.DataFrame(rows)[columns].tail(4).to_string(index=False))

      factor  value  best_epoch  final_train_accuracy  best_val_accuracy  last_epoch_val_accuracy  training_time_seconds
weight_decay 0.0000          19              1.000000            0.37475                  0.37350              43.905673
weight_decay 0.0001          19              1.000000            0.38250                  0.37950              20.387447
weight_decay 0.0010          10              1.000000            0.38875                  0.36875              22.540220
weight_decay 0.0100          17              0.770375            0.33425                  0.32625              19.724226


Finally, I saved all the results and the validation predictions from the best setting.

In [5]:
summary = pd.DataFrame(rows)
summary.to_csv(ROOT / 'results' / 'image' / 'nn_v9_summary.csv', index=False)

# Highest validation accuracy wins; a tie goes to less dropout, then less weight decay.
best = max(rows, key=lambda row: (row['best_val_accuracy'], -row['dropout'], -row['weight_decay']))
best_predictions = runs[(best['dropout'], best['weight_decay'])]['predictions']
validation = pd.DataFrame({
    'validation_index': np.arange(len(y_val)),
    'actual_label': y_val,
    'predicted_label': best_predictions,
})
validation.to_csv(ROOT / 'results' / 'image' / 'nn_v9_validation.csv', index=False)
saved = (validation['actual_label'] == validation['predicted_label']).mean()
print(f"best: dropout={best['dropout']}, weight_decay={best['weight_decay']}, "
      f"validation accuracy {best['best_val_accuracy']:.5f} (saved file gives {saved:.5f}, {len(validation)} rows)")

best: dropout=0.0, weight_decay=0.001, validation accuracy 0.38875 (saved file gives 0.38875, 4000 rows)


Regularization did not help. The V4 setting (weight decay 0.001, no dropout) was still the best at 0.389. Dropout 0.2 dropped to 0.375 and dropout 0.5 got 0.382. Weight decay 0.01 was too strong: training accuracy fell to 0.77 and validation fell to 0.334. Removing weight decay completely gave 0.375, so the small amount from V4 does help a little.

Even with dropout the training accuracy stayed above 99%, so the network still memorizes the training set. Dropout 0.5 was still improving at the last epoch (its best epoch was 20), so it might do better with more epochs, but I kept the epochs the same so only one thing changed.